In [27]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import json
import time
import xarray as xr
import scipy.signal as signal


import xgboost as xgb
import sklearn
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, StratifiedKFold, permutation_test_score, GridSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,make_scorer,balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt

data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
reference_dir = data_dir + '/CHCH_v3_reference/'
reference_mask_dir = reference_dir + 'processed{}/'
model_save_dir = reference_dir + 'models/'

### Loading model from file

In [22]:
model_infile = model_save_dir + 'model_001.json'

model = xgb.XGBClassifier();
model.load_model(model_infile)


In [3]:
model

XGBClassifier(base_score='5E-1', booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None,
              feature_types=['float', 'float', 'float', 'float', 'float', 'int',
                             'int', 'float'],
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

### Loading incoming data

In [14]:
date = '20210519'
EXCL_VARS = ['vdr', 'vdr_cm', 'wlwc', 'slwc', 'mpc', 'slcc', 'ic', 'undet', 'u_nan', 'unknown']

In [102]:
infile = reference_mask_dir.format(date[:-2])+str(date)+'.nc'
print('  <--- ' + infile)

ds = xr.open_dataset(infile, drop_variables=EXCL_VARS)

time = ds.time
backscatter = ds.backscatter.values
temp = ds.temp.values
cloud_mask = ds.cloud_mask
altitude = ds.zfull.values

ds.close()

  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210519.nc


### Initializing output dataset

In [127]:
ds_new = ds.copy()

ds_new['classified_peaks'] = xr.full_like(ds_new.backscatter, np.nan)

ds_new['wlwc'] = xr.zeros_like(ds_new.cloud_mask)
ds_new['slwc'] = xr.zeros_like(ds_new.cloud_mask)
ds_new['mpc'] = xr.zeros_like(ds_new.cloud_mask)
ds_new['ic'] = xr.zeros_like(ds_new.cloud_mask)

ds_new['wlwc_t'] = xr.zeros_like(ds_new.time, dtype='int8')
ds_new['slwc_t'] = xr.zeros_like(ds_new.time, dtype='int8')
ds_new['mpc_t'] = xr.zeros_like(ds_new.time, dtype='int8')
ds_new['ic_t'] = xr.zeros_like(ds_new.time, dtype='int8')

ds_new

<xarray.Dataset>
Dimensions:           (time: 288, level: 300)
Coordinates:
  * time              (time) datetime64[ns] 2021-05-19T00:02:29.999991 ... 20...
    zfull             (level) float64 25.0 75.0 125.0 ... 1.492e+04 1.498e+04
Dimensions without coordinates: level
Data variables:
    backscatter       (time, level) float64 2.772e-06 6.511e-07 ... -3.502e-07
    cloud_mask        (time, level) int8 0 0 0 1 1 1 1 1 1 ... 0 0 0 0 0 0 0 0 0
    temp              (time, level) float64 11.98 11.98 11.41 ... -59.16 -59.23
    classified_peaks  (time, level) float64 nan nan nan nan ... nan nan nan nan
    wlwc              (time, level) int8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    slwc              (time, level) int8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    mpc               (time, level) int8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    ic                (time, level) int8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    wlwc_t            (time) int8 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    slwc_t            (time) int8 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    mpc_t             (time) int8 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    ic_t              (time) int8 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

In [91]:
# ds_new['classified_peaks'][dict(time=10,level=299)]  = 0
# # ds_new['classified_peaks'][dict(time=10)][288:305]  = 1


# ds_new['classified_peaks'].isel(time=10)

<xarray.DataArray 'classified_peaks' (level: 300)>
array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0], dtype=int8)
Coordinates:
    time     datetime64[ns] 2021-05-19T00:52:29.999973
    zfull    (level) float64 ...
Dimensions without coordinates: level
Attributes:
    long_name:  cloud mask
    units:      1

### Classifying profiles sequentially

In [128]:
magnitude_threshold = 2e-5
width_threshold = 1

for time_idx, profile in enumerate(backscatter):
    
    ### For each vertical profile, gets peak properties
    peaks, properties = signal.find_peaks(profile,height=magnitude_threshold, width=width_threshold, rel_height=0.5, prominence=(None,None))

    for peak_idx, peak_alt_idx in enumerate(peaks):  
                      
            peak_dict = {                
                # Backscatter properties
                'peak_magnitude' : properties['peak_heights'][peak_idx],
                'peak_width' : properties['widths'][peak_idx],
                'peak_width_height' : properties['width_heights'][peak_idx],
                'peak_prominence' : properties['prominences'][peak_idx],
                'peak_alt' : altitude[peak_alt_idx]/1000,
                'num_peaks' : len(peaks),
                'peak_order' : peak_idx,
                
                # AMPS temperature
                'peak_temp' : temp[time_idx, peak_alt_idx],                                
                }
            
            peak_df = pd.DataFrame.from_dict([peak_dict])
            Z = model.predict(peak_df)[0] #0=WLWC, 1=SLWC, 2=MPC 3=IC

            min_alt = int(peak_alt_idx-properties['widths'][peak_idx])
            max_alt = int(peak_alt_idx+properties['widths'][peak_idx])
            
            min_idx = max(0,min_alt)
            max_idx = min(299,max_alt)

            ds_new['classified_peaks'][dict(time=time_idx,level=peak_alt_idx)] = Z

            if Z == 0:
                ds_new['wlwc_t'][dict(time=time_idx)] = 1
                ds_new['wlwc'][dict(time=time_idx)][min_idx:max_idx] = 1 
            elif Z == 1:
                ds_new['slwc_t'][dict(time=time_idx)] = 1
                ds_new['slwc'][dict(time=time_idx)][min_idx:max_idx] = 1
            elif Z == 2:
                ds_new['mpc_t'][dict(time=time_idx)] = 1
                ds_new['mpc'][dict(time=time_idx)][min_idx:max_idx] = 1
            elif Z == 3:
                ds_new['ic_t'][dict(time=time_idx)] = 1
                ds_new['ic'][dict(time=time_idx)][min_idx:max_idx] = 1
            

In [130]:
ds_new['slwc'].isel(time=10)

<xarray.DataArray 'slwc' (level: 300)>
array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8)
Coordinates:
    time     datetime64[ns] 2021-05-19T00:52:29.999973
    zfull    (level) float64 25.0 75.0 125.0 ... 1.488e+04 1.492e+04 1.498e+04
Dimensions without coordinates: level
Attributes:
    long_name:  cloud mask
    units:      1

## AG model

In [13]:
from sklearn.pipeline import Pipeline
import joblib

### Opens AG model
model_ag_load = xgb.XGBClassifier();
model_ag_load.load_model("models/AG_XGBoost.json")
std = joblib.load('models/AG_scaler.save') 

pipe = Pipeline([('standardscaler',std),('xgbclassifier',model_ag_load)])



/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [09:05:07] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1712072639327/work/src/learner.cc:873: Found JSON model saved before XGBoost 1.6, please save the model using current version again. The support for old JSON model will be discontinued in XGBoost 2.3.
  warnings.warn(smsg, UserWarning)


In [14]:
date = '20210519'
EXCL_VARS = ['vdr', 'vdr_cm', 'wlwc', 'slwc', 'mpc', 'slcc', 'ic', 'undet', 'u_nan', 'unknown']

infile = reference_mask_dir.format(date[:-2])+str(date)+'.nc'
print('  <--- ' + infile)

ds = xr.open_dataset(infile, drop_variables=EXCL_VARS)

time = ds.time
backscatter = ds.backscatter.values
temp = ds.temp.values
cloud_mask = ds.cloud_mask
altitude = ds.zfull.values

ds.close()

  <--- /home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v3_reference/processed202105/20210519.nc


In [16]:
ds_new = ds.copy()

ds_new['classified_peaks'] = xr.full_like(ds_new.backscatter, np.nan)

ds_new['slwc'] = xr.zeros_like(ds_new.cloud_mask)
ds_new['slwc_t'] = xr.zeros_like(ds_new.time, dtype='int8')

ds_new

<xarray.Dataset>
Dimensions:           (time: 288, level: 300)
Coordinates:
  * time              (time) datetime64[ns] 2021-05-19T00:02:29.999991 ... 20...
    zfull             (level) float64 25.0 75.0 125.0 ... 1.492e+04 1.498e+04
Dimensions without coordinates: level
Data variables:
    backscatter       (time, level) float64 2.772e-06 6.511e-07 ... -3.502e-07
    cloud_mask        (time, level) int8 0 0 0 1 1 1 1 1 1 ... 0 0 0 0 0 0 0 0 0
    temp              (time, level) float64 11.98 11.98 11.41 ... -59.16 -59.23
    classified_peaks  (time, level) float64 nan nan nan nan ... nan nan nan nan
    slwc              (time, level) int8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    slwc_t            (time) int8 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

In [20]:
magnitude_threshold = 2e-5
width_threshold = 1

### Need to reorder columns 
new_cols = ['peak_order','num_peaks','peak_prominence','peak_magnitude','peak_width','peak_width_height','peak_temp','peak_alt']


for time_idx, profile in enumerate(backscatter):
    
    ### For each vertical profile, gets peak properties
    peaks, properties = signal.find_peaks(profile,height=magnitude_threshold, width=width_threshold, rel_height=0.5, prominence=(None,None))

    for peak_idx, peak_alt_idx in enumerate(peaks):  
                    
            peak_dict = {                
                # Backscatter properties
                'peak_magnitude' : properties['peak_heights'][peak_idx],
                'peak_width' : properties['widths'][peak_idx],
                'peak_width_height' : properties['width_heights'][peak_idx],
                'peak_prominence' : properties['prominences'][peak_idx],
                'peak_alt' : altitude[peak_alt_idx]/1000,
                'num_peaks' : len(peaks),
                'peak_order' : peak_idx,
                
                # AMPS temperature
                'peak_temp' : temp[time_idx, peak_alt_idx],                                
                }
            
            peak_df = pd.DataFrame.from_dict([peak_dict])
            peaks_df_ag = peak_df[new_cols]
            Z = pipe.predict(peaks_df_ag)[0]

            min_alt = int(peak_alt_idx-properties['widths'][peak_idx])
            max_alt = int(peak_alt_idx+properties['widths'][peak_idx])
            
            min_idx = max(0,min_alt)
            max_idx = min(299,max_alt)

            ds_new['classified_peaks'][dict(time=time_idx,level=peak_alt_idx)] = Z

            if Z == 1:
                ds_new['slwc_t'][dict(time=time_idx)] = 1
                ds_new['slwc'][dict(time=time_idx)][min_idx:max_idx] = 1

            
ds_new         

/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/lukeew/anaconda3/envs/01_SLW_xgb/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/lukeew/anaconda3/envs/01_SLW_xgb/l

<xarray.Dataset>
Dimensions:           (time: 288, level: 300)
Coordinates:
  * time              (time) datetime64[ns] 2021-05-19T00:02:29.999991 ... 20...
    zfull             (level) float64 25.0 75.0 125.0 ... 1.492e+04 1.498e+04
Dimensions without coordinates: level
Data variables:
    backscatter       (time, level) float64 2.772e-06 6.511e-07 ... -3.502e-07
    cloud_mask        (time, level) int8 0 0 0 1 1 1 1 1 1 ... 0 0 0 0 0 0 0 0 0
    temp              (time, level) float64 11.98 11.98 11.41 ... -59.16 -59.23
    classified_peaks  (time, level) float64 nan nan nan nan ... nan nan nan nan
    slwc              (time, level) int8 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    slwc_t            (time) int8 0 1 0 0 1 0 1 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

In [32]:
# type(pipe) == sklearn.pipeline.Pipeline
# type(model) == xgb.sklearn.XGBClassifier
